# Fine-Tuning Google's T5 for Dialogue Summarization

This notebook provides a complete pipeline to fine-tune the **T5-Small** transformer model on the **SAMSum dataset** (a conversational dialogue dataset with human-annotated summaries).

### Workflow Overview:
1. **Environment Setup**: Installing the `transformers[torch]` library.
2. **Data Loading & Sampling**: Loading the train/validation datasets and sub-sampling them to manage runtimes.
3. **Data Preprocessing**: Cleaning up conversational text (removing formatting tags, handling line breaks).
4. **Tokenization**: Mapping text inputs and outputs to token IDs.
5. **Model & Hardware Acceleration**: Initializing the T5 model and configuring PyTorch device placement (GPU/CPU).
6. **Fine-Tuning**: Setting up training parameters and running the Hugging Face `Trainer`.
7. **Saving and Google Drive Integration**: Persisting weights locally and backing them up to Drive storage.
8. **Inference pipeline**: Running a helper function to summarize custom conversation text.

In [1]:
!pip install "transformers[torch]"

## 🛠️ Step 1: Import Dependencies
We import data manipulation tools (`pandas`), regex (`re`), and the required Hugging Face transformer components for model training and tokenizer handling.

In [2]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration , Trainer , TrainingArguments

## 📂 Step 2: Load the SAMSum Dataset
Here, we load the dialogue train and validation datasets from CSV files. Make sure the dataset CSVs are placed in the path `/content/` when running on Google Colab, or in the current directory if running locally.

In [ ]:
train_data=pd.read_csv('/content/samsum-train.csv')
val_data = pd.read_csv('/content/samsum-validation.csv')


### Inspect Dataset Structure
Let's look at the first few rows of the training dataset. We have `id`, `dialogue` (the messenger logs), and `summary` (the human-written summaries).

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

### Dataset Sub-sampling
The SAMSum dataset contains over 14,000 training examples. To keep execution fast and prevent memory issues on typical notebook environments, we sample **4,000 training instances** and **500 validation instances**.

In [ ]:
# random samppling
train_data =train_data.sample(n=4000 , random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500 , random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

## 🧹 Step 3: Text Cleaning & Preprocessing
Raw dialogues often contain messy line breaks (`\r\n`), multiple concurrent spaces, or empty HTML tags. We define a helper function to replace these anomalies with clean, normalized spacing.

In [ ]:
import re

def clean_data(text):
  text = re.sub(r"\r\n" , " " , text) #lines
  text = re.sub(r"\s+" , " " , text) #spaces
  text = re.sub(r"<.*?>" , " " , text) #html tag * means any <p> , <body> etc
  text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


In [ ]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

## 🔠 Step 4: Tokenization & Training Prep
We instantiate Google's pretrained `t5-small` tokenizer. We will use this to convert our text strings into token IDs.

In [3]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

### Define Tokenization Function
This helper maps input dialogues and output summaries to token IDs. 
- Inputs are padded and truncated to a maximum length of **512 tokens**.
- Target summaries are padded and truncated to **150 tokens** and mapped as the `labels` key for training.

In [ ]:
# raw data=> tokeined input for fine tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"] , padding="max_length" , max_length=512 , truncation=True)
  targets = tokenizer(data["summary"] , padding="max_length" , max_length=150 , truncation=True)

  inputs["labels"] = targets["input_ids"] #tokens ids => add to inputs as labels
  return inputs




In [ ]:
train_dataset = train_data.apply(tokenize , axis=1).tolist()
val_dataset = val_data.apply(tokenize , axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
#input ids - dialogue => token ids
# 1=> EOS

# attention mask
#labels - target => summary token

In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:
type(train_dataset)

list

In [ ]:
type(val_dataset)

list

## 🚀 Step 5: Initialize T5 Model & Hardware Acceleration
Next, we initialize the model weights using the pre-trained `t5-small` sequence-to-sequence model.

In [4]:
#NLP => generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Device Setup
We detect the available hardware runtime (CUDA GPU, Apple Silicon MPS, or CPU) and move the model parameters to it for faster training iterations.

In [ ]:
import torch
if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
   device = torch.device("cpu")
print("device: " , device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

### Device Setup
We detect the available hardware runtime (CUDA GPU, Apple Silicon MPS, or CPU) and move the model parameters to it for faster training iterations.

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA Available: True
GPU: Tesla T4


In [ ]:
!nvidia-smi

Thu Jul 23 07:48:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             27W /   70W |     343MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 🏋️ Step 6: Configure Training Arguments
We define the training hyperparameters using `TrainingArguments`. This sets up the learning rate (`3e-4`), batch sizes (`8`), weight decay, number of epochs (`4`), and enables mixed precision (`fp16`) to speed up execution on GPUs like the NVIDIA Tesla T4.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=4,

    learning_rate=3e-4,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=200,

    fp16=True,              # Use mixed precision on T4 GPU
    logging_steps=50,

    report_to="none",
)

### Device Setup
We detect the available hardware runtime (CUDA GPU, Apple Silicon MPS, or CPU) and move the model parameters to it for faster training iterations.

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


### Instantiate Hugging Face Trainer
We pass the model, training arguments, tokenized datasets, and eval datasets to the standard HF `Trainer` interface.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

### Fine-Tune the Model
We trigger the training run. The output table displays training loss, evaluation loss, and learning rate adjustments epoch by epoch.

In [ ]:
# train model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.388512,0.357063
2,0.345916,0.347199
3,0.326641,0.346364
4,0.306306,0.345886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=0.6027931084632874, metrics={'train_runtime': 454.4841, 'train_samples_per_second': 35.205, 'train_steps_per_second': 4.401, 'total_flos': 2165468823552000.0, 'train_loss': 0.6027931084632874, 'epoch': 4.0})

## 💾 Step 7: Local Model Saving & Verification
After the training completes, we save the fine-tuned model and tokenizer weights to a local directory `./saved_summary_model`.

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

### Verify Model Loading
We double check that the weights load back correctly by reloading the model and tokenizer from our local path.

In [ ]:
model.from_pretrained("./saved_summary_model")
tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5Tokenizer(name_or_path='./saved_summary_model', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False

## ☁️ Step 8: Google Drive Mount & Backup (Colab Only)
Google Colab runtimes discard local files when a session ends. To persist our trained model permanently, we mount our Google Drive workspace at `/content/drive`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

save_path = "/content/drive/MyDrive/Text_Summarizer_Model"

os.makedirs(save_path, exist_ok=True)

### Save Weights to Drive Folder
We define a folder path `/content/drive/MyDrive/Text_Summarizer_Model` inside our Google Drive and export the model state and tokenizer files there for future deployments.

In [ ]:
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Text_Summarizer_Model/tokenizer_config.json',
 '/content/drive/MyDrive/Text_Summarizer_Model/tokenizer.json')

## 🔮 Step 9: Inference Pipeline
To summarize dialogues on demand, we reload the model weights and set up a custom inference pipeline.

In [6]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_path = "/content/savedmodel"

tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded successfully!


### Create Summarization Helper Function
This helper runs inference on raw dialogue strings. It tokenizes the input, calls `model.generate` using **Beam Search** with `num_beams=4` for high-quality text output, and decodes the resulting tokens back to regular text.

In [7]:
def summarize_dailogue(dialogue):
  #tokenize
  input=tokenizer(dialogue ,
  padding="max_length" ,
  max_length=512 ,
  truncation=True ,
  return_tensors="pt")

  #generate  the summary = >  token ids
  targets = model.generate(
    input_ids = input["input_ids"],
    attention_mask = input["attention_mask"],
    max_length=150,
    num_beams=4,
    early_stopping=True
  )

  # decoded our output
  summary = tokenizer.decode(targets[0] , skip_special_tokens=True)#EOS,SEP
  return summary







### Test Inference
Let's test the trained model on a sample technology news reporter conversation to inspect the quality of the generated summary.

In [11]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dailogue(test_dialogue)
print("Summary:", summary)

Summary: In today's technology news, artificial intelligence continues to expand across industries, from healthcare to finance and education. Companies investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets.
